# 14 — Evidence-Grounded Application

## Objective
Build and verify a transparent biomedical evidence object before generating a structured interpretation.

## Background
A language model should explain evidence computed by deterministic statistical tools, not invent biomarker statistics. TCGA (The Cancer Genome Atlas) LUAD (Lung Adenocarcinoma) is a possible public-data integration source; this notebook demonstrates the evidence-to-report interface without claiming it downloaded a cohort.

### Demonstrate an evidence interpretation contract

**Procedure.** Run a deterministic interpretation of explicitly illustrative survival statistics and return a structured evidence summary.

**Reading the result.** The report rejects unsupported causal inference. Its hazard ratio and confidence interval are synthetic teaching values, not TCGA findings.

In [1]:
# This evidence is synthetic teaching material, not a measured cohort result.
import json
import math
evidence = {"cohort": "DEMONSTRATION ONLY", "gene": "GENE_X", "illustrative": True,
            "hazard_ratio": 1.4, "confidence_interval": [0.9, 2.2],
            "p_value": 0.12, "replicated": False}

def assess(item):
    """Validate numeric inputs and keep synthetic evidence out of factual claims."""
    required = {"cohort", "gene", "illustrative", "hazard_ratio",
                "confidence_interval", "p_value", "replicated"}
    if not required.issubset(item):
        raise ValueError("Evidence object lacks required provenance or statistical fields.")
    if not isinstance(item["illustrative"], bool) or not isinstance(item["replicated"], bool):
        raise TypeError("Illustrative and replicated must be Boolean flags.")
    if not isinstance(item["gene"], str) or not item["gene"].strip():
        raise ValueError("Gene identifier is required.")
    if not isinstance(item["cohort"], str) or not item["cohort"].strip():
        raise ValueError("Cohort identifier is required.")
    values = [item["hazard_ratio"], item["p_value"], *item["confidence_interval"]]
    if any(isinstance(value, bool) or not isinstance(value, (int, float))
           or not math.isfinite(value) for value in values):
        raise ValueError("Hazard ratio, p-value, and interval must be finite numbers.")
    ratio, p_value = item["hazard_ratio"], item["p_value"]
    low, high = item["confidence_interval"]
    if ratio <= 0 or low <= 0 or low > ratio or ratio > high or not 0 <= p_value <= 1:
        raise ValueError("Invalid hazard ratio, confidence interval, or p-value.")
    statistical_criterion_met = (low > 1 or high < 1) and p_value < 0.05
    return {"gene": item["gene"],
            "survival_association_supported": bool(statistical_criterion_met and not item["illustrative"]),
            "replicated": item["replicated"], "causal_claim_supported": False,
            "limitations": (["Illustrative evidence; not a TCGA result"] if item["illustrative"] else [])
                           + ["Observational association does not establish causation"]}

report = assess(evidence)
print(json.dumps(report, indent=2))
assert report["survival_association_supported"] is False
assert report["causal_claim_supported"] is False

{
  "gene": "GENE_X",
  "survival_association_supported": false,
  "replicated": false,
  "causal_claim_supported": false,
  "limitations": [
    "Illustrative evidence; not a TCGA result",
    "Observational association does not establish causation"
  ]
}


### Immediate interpretation of the evidence report

The `GENE_X` values are invented. The reported association and causal-claim flags are both false; even a numerically significant synthetic combination is barred from a factual association claim. No cohort was analyzed.

### Validate the structured evidence response

**Procedure.** Check the required report fields and the limitations data type.

**Reading the result.** The successful message means the example conforms to a minimal schema; it does not establish factual or clinical correctness.

In [2]:
# Check the example's response shape and the boundary for synthetic evidence.
required = {"gene", "survival_association_supported", "replicated",
            "causal_claim_supported", "limitations"}
assert required.issubset(report)
assert isinstance(report["limitations"], list)
synthetic_significant = {**evidence, "hazard_ratio": 1.5,
                         "confidence_interval": [1.2, 1.8], "p_value": 0.01}
assert assess(synthetic_significant)["survival_association_supported"] is False
try:
    assess({**evidence, "confidence_interval": [2.2, 0.9]})
except ValueError:
    pass
else:
    raise AssertionError("Malformed confidence intervals must be rejected.")
print("Structured evidence schema and illustrative-data boundary: valid.")

Structured evidence schema and illustrative-data boundary: valid.


## Retrieval-Augmented Generation (RAG): evidence-grounded response design

**Objective.** Connect the post-trained biomedical model to traceable external evidence without assuming that retrieved text is automatically true or relevant.

**How to reduce unsupported answers:** retrieve a bounded set of source passages, preserve stable identifiers, rerank for topic relevance, instruct the model to cite passage identifiers, validate that reported claims appear in the evidence, and abstain when the evidence is insufficient. A separate verification step should test claim support; citations are not proof unless the cited passage actually supports the statement. Guard against source text that contains malicious instructions.

**How to respond to irrelevant vector-search results:** first log the query and returned identifiers; inspect the embedding domain, chunk size, source metadata, search filters, and index freshness. Try keyword or **BM25 (Best Matching 25)** retrieval, hybrid lexical–dense retrieval, domain-appropriate embeddings, reranking, and score-based rejection calibrated on labeled queries. Do not simply increase top-k until an answer appears.

The following cell is a **deterministic teaching model** that uses token overlap to select an example passage. It is *not* a vector database, a live literature search, or an LLM evaluation. In production, replace the illustrative passage store with indexed, rights-compliant source data and assess retrieval quality using notebook 04.

In [3]:
# Purpose: Demonstrate source-ID preservation, simple retrieval, and abstention without an LLM.
# Result: The first query finds a matching invented source; the unrelated query abstains.
import re

passages = [
    {"id": "DEMO-1", "text": "Hazard ratios describe relative event rates, not proof of causation."},
    {"id": "DEMO-2", "text": "A confidence interval crossing one may be compatible with no association."},
    {"id": "DEMO-3", "text": "A recipe describes how to bake bread."},
]

def terms(text):
    # Retain substantive tokens; this minimal matcher is deliberately not a production retriever.
    stop = {"a", "an", "and", "are", "do", "does", "how", "is", "of", "the", "to", "what"}
    return set(re.findall(r"[a-z]+", text.lower())) - stop

def retrieve(query, minimum_overlap=2):
    # A minimum overlap avoids presenting a weakly matched citation as strong evidence.
    hits = [(len(terms(query) & terms(p["text"])), p) for p in passages]
    score, best = max(hits, key=lambda value: value[0])
    return best if score >= minimum_overlap else None

for question in ["Do hazard ratios prove causation?", "Where are mitochondrial ribosomes translated?"]:
    hit = retrieve(question)
    answer = {"question": question, "source": hit["id"] if hit else None,
              "status": "source available; interpretation requires verification" if hit
                        else "abstain: no relevant source"}
    print(answer)
assert retrieve("Do hazard ratios prove causation?")["id"] == "DEMO-1"
assert retrieve("Where are mitochondrial ribosomes translated?") is None

{'question': 'Do hazard ratios prove causation?', 'source': 'DEMO-1', 'status': 'source available; interpretation requires verification'}
{'question': 'Where are mitochondrial ribosomes translated?', 'source': None, 'status': 'abstain: no relevant source'}


### Immediate interpretation of retrieval output

The first invented query matches passage `DEMO-1` by token overlap, while the unrelated query abstains. This checks source-ID propagation and a simple rejection threshold. It does not establish passage relevance in a real corpus or verify a generated answer.

## Agent workflows, LangChain, LangGraph, and the Model Context Protocol (MCP)

**Objective.** Explain when to use a sequential chain versus a stateful graph, how agents choose tools, and what a secure connection to external **Application Programming Interfaces (APIs)** requires.

**LangChain versus LangGraph.** A simple LangChain chain is appropriate for a predictable, mostly linear sequence such as retrieve → format prompt → generate → validate. LangGraph is preferable when steps branch, tools are retried, human approval interrupts the flow, multiple agents collaborate, or execution must resume after failure. LangGraph's durable execution uses persisted graph-state checkpoints associated with a `thread_id`; resume the same thread with a persistent checkpointer. Design side effects to be idempotent and preserve stable operation identifiers. Do not equate in-memory chat history with a crash-safe checkpoint.

**Agent tool selection.** A model can predict a structured tool call from available tool names, argument schemas, task context, and tool descriptions. A dispatcher validates the selected tool and its arguments before execution. The model does not grant permissions. A deterministic router is preferable when the task-to-tool mapping is straightforward.

**MCP (Model Context Protocol).** A host application connects through a client to one or more MCP servers. Servers expose tools, resources, and prompts. To connect an external API, implement or configure an MCP server that wraps a narrowly scoped API operation with an input schema, then connect it to the host using an authorized local or remote transport. Validate authentication, authorization, tool arguments, error handling, and output provenance. MCP is an interoperability protocol, not a security layer or an agent-planning algorithm.

**Single versus multiple agents.** Start with one agent if the task has a single coherent goal, shared context, and few tools. Introduce specialized agents only when responsibilities can be separated, independent verification is valuable, or parallel sub-tasks justify the coordination overhead. Measure total task success, cost, latency, and compounded errors before adopting multiple agents.

**Loop prevention.** Put step, token, time, and tool-call budgets outside the model. Detect repeating state/action pairs and lack of progress; stop with an explicit failure status. LangGraph also provides iteration/recursion limits; external watchdogs remain useful for long-running tools.

See [15 — Reproducibility and Packaging](15_mlops_pipeline.ipynb) for a small durable-state and monitoring exercise and [16 — Technical Decisions](16_technical_discussion.ipynb) for the full Q&A.

References: [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence) · [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture).

In [4]:
# Purpose: Demonstrate schema-checked tool selection and bounded steps without an agent framework.
# Result: Permitted calculations succeed, unknown tools are rejected, and repeated calls stop.
#         The inputs and results are illustrative, not clinical evidence.
from collections import Counter

allowed_tools = {
    "read_evidence": lambda arguments: {"id": "DEMO-1", "text": passages[0]["text"]},
    "multiply": lambda arguments: arguments["a"] * arguments["b"],
}

def dispatch(tool_name, arguments):
    # Reject unknown operations even if a language model requests them.
    if tool_name not in allowed_tools:
        raise ValueError("Tool is not allowlisted")
    if tool_name == "multiply":
        if set(arguments) != {"a", "b"} or not all(
            isinstance(arguments[k], (int, float)) and not isinstance(arguments[k], bool)
            for k in ("a", "b")
        ):
            raise ValueError("multiply requires numeric a and b")
    elif arguments:
        raise ValueError("read_evidence accepts no arguments")
    return allowed_tools[tool_name](arguments)

print("Illustrative calculation:", dispatch("multiply", {"a": 2, "b": 3}))
try:
    dispatch("execute_shell", {"command": "echo hello"})
except ValueError as exc:
    print("Blocked:", exc)

# Prevent repetition using an application-enforced action budget.
proposed_tools = ["read_evidence"] * 8
calls = Counter()
for step, tool_name in enumerate(proposed_tools, start=1):
    if step > 5 or calls[tool_name] >= 3:
        print("Stopped: step or repetition budget reached at step", step)
        break
    calls[tool_name] += 1
    dispatch(tool_name, {})
assert calls["read_evidence"] == 3

Illustrative calculation: 6
Blocked: Tool is not allowlisted
Stopped: step or repetition budget reached at step 4


### Results

The invented `GENE_X` report kept survival-association and causal-claim flags false, including for a nominally significant synthetic combination. A reversed confidence interval was rejected. The token-overlap example retrieved invented passage `DEMO-1` and abstained on an unrelated question; the dispatcher blocked an unallowlisted shell call and stopped repeated reads.

### Interpretation

These checks exercise schema validation, conservative evidence flags, simple retrieval, and bounded tool routing.

### Limitations

No TCGA cohort analysis, live OncoLncAI call, model-generated evidence report, vector database, or service security test occurred. Invented statistics cannot support a biomedical finding.

### Key Observations

The notebook is an illustrative interface demonstration, not an implemented clinical integration.


### Real-data extension
If you execute the optional TCGA-LUAD study, record the exact Xena dataset identity and processing method. Perform valid expression/survival analysis separately, serialize source-grounded results, and only then pass those results to model inference. Never infer a hazard ratio from an LLM completion.

## Key observations and technical discussion
The example verifies an interface and an evidence-boundary rule, not biological discovery. Any actual clinical claim needs validated clinical evidence and appropriate expertise.

## Q&A
**Question:** Why should a language model not invent a hazard ratio from narrative context?

**Answer:** The numeric estimate must come from a traceable statistical analysis, not from text prediction.

### Reproducibility note
Each printed example belongs to one of three categories: an illustrative unit check, a source-backed inspection, or an opt-in model experiment. Only real executed experiments with source/model identifiers and saved outputs can support research findings.